# Hourly Energy Load Prediction

**Tabular Regression Project** — Predict hourly electrical energy load (MW) from weather and temporal features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `energy_load_mw`

## 2 · Project Overview

We predict **hourly electrical energy load** (in megawatts) using weather conditions, temporal patterns, and regional characteristics. Energy load forecasting is fundamental for grid operators, power markets, and renewable energy integration.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given the hour of day, weather (temperature, humidity, wind, cloud cover), calendar info, population density, and industrial activity index, predict the **grid energy load in MW** for that hour.

## 5 · Why This Project Matters

- **Grid stability** requires accurate load forecasts hours to days ahead.
- **Energy markets** use demand forecasts for pricing and trading.
- **Renewable integration** depends on demand-supply matching.
- **Peak shaving** and demand response programs need reliable predictions.
- A core problem in smart grid and energy analytics.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | 10 (hour, temperature, humidity, wind, cloud cover, day of week, holiday, population density, industrial index, month) |
| **Target** | `energy_load_mw` (continuous, megawatts) |
| **Categorical** | None (all numeric) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "energy_load_mw"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 8000
hour = np.random.randint(0, 24, n)
temperature = np.random.uniform(-10, 42, n)
humidity = np.random.uniform(15, 95, n)
wind_speed = np.random.uniform(0, 40, n)
cloud_cover = np.random.uniform(0, 100, n)
dow = np.random.randint(0, 7, n)
is_holiday = np.random.choice([0, 1], n, p=[0.92, 0.08])
pop_density = np.random.uniform(50, 5000, n)
industrial_idx = np.random.uniform(0.1, 1.0, n)
month = np.random.randint(1, 13, n)

# Energy load pattern: high during business hours, HVAC load from extreme temps
hour_load = 200 * np.exp(-0.5 * ((hour - 14) / 5) ** 2)  # afternoon peak
temp_load = 15 * np.abs(temperature - 20)  # HVAC: heating below 20, cooling above 20
weekend_effect = np.where(dow >= 5, -150, 0).astype(float)
season_load = np.where((month >= 6) & (month <= 8), 100,
              np.where((month == 12) | (month <= 2), 80, 0)).astype(float)

load = (500 + hour_load + temp_load + pop_density * 0.1
        + industrial_idx * 300 - humidity * 0.5
        + weekend_effect + season_load - is_holiday * 100
        + np.random.normal(0, 50, n))
load = np.maximum(load, 100)

df = pd.DataFrame({
    "hour": hour, "temperature_c": temperature, "humidity_pct": humidity,
    "wind_speed_kmh": wind_speed, "cloud_cover_pct": cloud_cover,
    "day_of_week": dow, "is_holiday": is_holiday,
    "population_density": pop_density, "industrial_index": industrial_idx,
    "month": month, "energy_load_mw": load
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["temperature_c", "humidity_pct", "population_density",
                          "industrial_index", "cloud_cover_pct", "wind_speed_kmh"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["energy_load_mw"], alpha=0.2, s=8)
    ax.set_xlabel(col); ax.set_ylabel("energy_load_mw"); ax.set_title(f"Load vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("hour")["energy_load_mw"].mean().plot(ax=axes[0], marker="o", color="steelblue")
axes[0].set_title("Mean Energy Load by Hour"); axes[0].set_ylabel("MW")
df.groupby("month")["energy_load_mw"].mean().plot(ax=axes[1], marker="s", color="darkorange")
axes[1].set_title("Mean Energy Load by Month"); axes[1].set_ylabel("MW")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `energy_load_mw`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel(TARGET)
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: {df[TARGET].mean():,.1f} MW, Median: {df[TARGET].median():,.1f} MW")
print(f"Std: {df[TARGET].std():,.1f} MW, Range: {df[TARGET].min():,.1f} - {df[TARGET].max():,.1f} MW")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

# Cyclical hour and month encoding
for df_part in [X_train, X_test]:
    df_part["hour_sin"] = np.sin(2 * np.pi * df_part["hour"] / 24)
    df_part["hour_cos"] = np.cos(2 * np.pi * df_part["hour"] / 24)
    df_part["month_sin"] = np.sin(2 * np.pi * df_part["month"] / 12)
    df_part["month_cos"] = np.cos(2 * np.pi * df_part["month"] / 12)

# HVAC proxy: deviation from comfort temperature (20°C)
X_train["temp_deviation"] = np.abs(X_train["temperature_c"] - 20)
X_test["temp_deviation"] = np.abs(X_test["temperature_c"] - 20)

# Is weekend
X_train["is_weekend"] = (X_train["day_of_week"] >= 5).astype(int)
X_test["is_weekend"] = (X_test["day_of_week"] >= 5).astype(int)

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Population density** is the top predictor — more people = more electricity consumption.
- **Hour of day** shows a clear afternoon peak (business hours + cooling demand).
- **Temperature deviation from 20°C** drives HVAC load — both extreme heat and cold increase demand.
- **Industrial index** contributes a steady baseline load.
- **Weekends and holidays** reduce load by ~150 MW.

**Business takeaway:** Focus demand-response programs on weekday afternoons in extreme weather. Industrial users provide predictable baseload.

## 26 · Limitations

1. Synthetic data — real grid load has complex spatial and temporal dynamics.
2. No lag features — past load is the best predictor of future load.
3. Population density is static — real demand shifts with daily commute patterns.
4. No renewable generation data — net load matters more than gross load.
5. Single-region model — real grids have zonal load patterns.

## 27 · How to Improve This Project

1. Add real utility load data (PJM, ERCOT, CAISO).
2. Include lag features (load at t-1, t-24, t-168).
3. Add renewable generation as a feature for net load prediction.
4. Use time-series models (LSTM, Transformer) for sequential patterns.
5. Model at 15-minute granularity for demand response applications.

## 28 · Production Considerations

- Integrate with weather forecast APIs for next-day load prediction.
- Provide probabilistic forecasts for risk management.
- Monitor forecast errors in real-time and trigger alerts.
- Retrain with recent load data to capture evolving demand patterns.

## 29 · Common Mistakes

1. Ignoring the non-linear temperature-load relationship (V-shaped HVAC curve).
2. Treating hour and month as linear features instead of cyclical.
3. Not separating weekday and weekend patterns.
4. Using current weather instead of weather forecasts for production deployment.
5. Ignoring the autocorrelation of load — yesterday's load predicts today's.

## 30 · Mini Challenge / Exercises

1. Remove `temp_deviation` and keep only `temperature_c` — does R² drop?
2. Add `temperature_c²` instead of the absolute deviation — compare performance.
3. Train separate models for weekdays and weekends — any improvement?
4. Use only hour + temperature + population — how close to the full model?
5. Try a polynomial regression (degree 2) as an enhanced baseline.

## 31 · Final Summary / Key Takeaways

1. **Population density** and **hour of day** are the strongest load predictors.
2. **Temperature** has a V-shaped relationship with load via HVAC.
3. **Cyclical encoding** of time features benefits linear models.
4. **Weekend/holiday effects** are significant and consistent.
5. **Gradient boosting** captures the non-linear temperature-load relationship well.
6. **Lag features** would be the single biggest improvement for production.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))